# Cleaning up dataset_completed.csv

Before I use this file for anything else, I went through it column by column
to check for messy stuff: duplicate categories written differently, rows
that look reachable but actually have no real data in them, weird logic
(like trackers going UP after clicking reject), and columns that don't
carry any real info. This notebook fixes what I found and saves a clean
version.


In [1]:
import pandas as pd

df = pd.read_csv("../clean_data/dataset_completed.csv")
print("starting shape:", df.shape)


starting shape: (4101, 31)


## 1. Same thing, written two different ways

I found a few spots where the exact same real-world value got saved under
two different spellings, probably because the scraper/API returned slightly
different text at different times. If I don't fix these, they'll show up as
"different" categories in my SQL queries and charts, which is wrong.


In [2]:
# "fundingchoices" and "funding choices" are the same CMP vendor (Google
# Funding Choices), just written with or without a space. Same for "commanders
# act" style naming vs "trustcommander" -- but I'm NOT touching that one,
# since I'm not 100% sure they're really the same product and don't want to
# merge two things that might actually be different.
df["cmp_detected"] = df["cmp_detected"].replace({"funding choices": "fundingchoices"})

# "Netherlands" and "The Netherlands" are literally the same country -- I
# checked and both have country_code = 'NL', so this one's an easy merge.
df["country"] = df["country"].replace({"The Netherlands": "Netherlands"})

# same idea: "Riga" and "Rīga" are the same city, just with/without the accent
df["region"] = df["region"].replace({"Rīga": "Riga"})

print("cmp_detected values now:", sorted(df["cmp_detected"].dropna().unique()))


cmp_detected values now: ['axeptio', 'commanders act', 'complianz', 'conversant', 'cookie notice', 'cookiebot', 'cookieyes', 'didomi', 'fundingchoices', 'iubenda', 'klaro', 'none', 'onetrust', 'osano', 'quantcast', 'setupad', 'sourcepoint', 'trustarc', 'trustcommander', 'usercentrics']


## 2. A row that says "reachable" but has nothing in it

I checked every row where `reachable` is True, and found ONE (`laredoute.fr`)
where literally every single scraped signal -- trackers, cookies, banner
info, all of it -- is empty. The geolocation part worked fine for this row,
so it's not that the whole site failed, just that the actual scraping part
silently didn't get anything. Since there's nothing usable left in this row
for any kind of analysis, I'm dropping it instead of letting it quietly turn
into a bunch of empty cells later.


In [3]:
before = len(df)
df = df[df["url"] != "https://laredoute.fr"].copy()
print(f"dropped {before - len(df)} row(s), new shape: {df.shape}")


dropped 1 row(s), new shape: (4100, 31)


## 3. Trackers going UP after clicking reject (shouldn't happen)

For sites where I actually tested clicking "reject" and reloading the page,
`tracker_count_after_reject` should logically be the same or lower than
`tracker_count`. I found 58 rows where it's actually HIGHER, which doesn't
make sense if reject really worked. My guess is this is just noise from
reloading the page (ads/scripts loading at different speeds each time), not
a real error. I'm not deleting these rows -- they're still real
measurements -- but I'm adding a flag column so I (or the modeling notebook)
can easily see which ones are noisy instead of trusting them blindly.


In [4]:
df["reject_test_looks_noisy"] = df["tracker_count_after_reject"] > df["tracker_count"]
print("flagged as noisy:", df["reject_test_looks_noisy"].sum(), "rows")


flagged as noisy: 58 rows


## 4. Columns that aren't actually useful

`reachable` is True for every single row in this file -- makes sense,
since the scraper already only kept sites it could reach before saving this
file. That means the column never changes, so it's not telling me anything.
I'm dropping it to keep things tidy.


In [5]:
print("unique values in reachable:", df["reachable"].unique())
df = df.drop(columns=["reachable"])
print("columns now:", df.shape[1])


unique values in reachable: [ True]
columns now: 31


## 5. Making the True/False columns actually behave like True/False

A bunch of columns (`has_reject_button`, `is_hosting`, etc.) are stored as
generic "object" type instead of a real boolean type. That's just because
they mix True/False with some empty (NaN) values, and plain pandas booleans
can't hold empty values. I'm switching them to pandas' "nullable boolean"
type, which can hold True/False/empty properly instead of a loose mix of
Python objects.


In [6]:
bool_cols = [
    "tcf_api_present", "has_accept_button", "has_reject_button", "has_cookie_banner",
    "reject_as_visible_as_accept", "prechecked_boxes_detected", "reject_click_attempted",
    "reject_click_succeeded", "has_privacy_policy", "robots_txt_present",
    "is_hosting", "flagged_unsafe", "reject_test_looks_noisy",
]
for c in bool_cols:
    df[c] = df[c].astype("boolean")

print(df[bool_cols].dtypes)


tcf_api_present                boolean
has_accept_button              boolean
has_reject_button              boolean
has_cookie_banner              boolean
reject_as_visible_as_accept    boolean
prechecked_boxes_detected      boolean
reject_click_attempted         boolean
reject_click_succeeded         boolean
has_privacy_policy             boolean
robots_txt_present             boolean
is_hosting                     boolean
flagged_unsafe                 boolean
reject_test_looks_noisy        boolean
dtype: object


## 6. Stuff I checked but decided NOT to change

A few things looked suspicious at first but turned out to be fine, so I'm
leaving them as they are:
- `org` is missing for about 6% of rows -- that's just because some
  internet providers don't report a separate "org" name, not a mistake.
- 6 rows have no geolocation info at all (country, isp, lat/lon all empty)
  -- the location lookup just failed for those specific sites. Everything
  else in those rows is fine, so I'm keeping them and just living with the
  gap.
- Negative numbers in `lat` and `lon` are normal, not errors -- that's just
  how you write coordinates for the southern half (lat) or western half
  (lon) of the world.
- `reject_as_visible_as_accept`, `tracker_count_after_reject` and
  `non_essential_cookies_after_reject` are empty for most rows -- but that's
  expected, since they only get filled in when a site actually HAD a reject
  button that got clicked. Nothing to fix there.


## Save the clean version


In [7]:
df.to_csv("../clean_data/dataset_completed_clean.csv", index=False)
print("saved dataset_completed_clean.csv --", df.shape)


saved dataset_completed_clean.csv -- (4100, 31)
